# PyTorch Training with Checkpointing and Preemption Handling

In this notebook, we will demonstrate:
 * PyTorch training with automatic checkpointing
 * RayJob preemption/suspension handling
 * Training resume from checkpoints
 * Both existing cluster and lifecycled cluster scenarios
 * Queue utilization analysis for resource optimization


## Import Required Packages

First, let's import the necessary CodeFlare SDK packages and PyTorch dependencies.


In [ ]:
from codeflare_sdk import (
    Cluster, 
    ClusterConfiguration, 
    RayJob, 
    TokenAuthentication
)
import time
import json
import os


## PyTorch Training Script

The PyTorch training script with checkpointing and preemption handling is now available as a separate Python file (`pytorch_training_with_checkpointing.py`). This makes it reusable and keeps the notebook focused on demonstrating the CodeFlare SDK usage.


The PyTorch training script focuses purely on checkpointing
Preemption handling is demonstrated in the notebook using RayJob suspension
✅ PyTorch training script is available as: pytorch_training_with_checkpointing.py
📋 The script includes:
- Complete CNN implementation
- Automatic checkpoint saving every epoch
- Automatic checkpoint loading and resume
- GPU/CPU device detection
- Clean separation: checkpointing in script, preemption in notebook


## Authentication

Set up authentication for accessing the cluster resources.


In [ ]:
# Create authentication object
auth = TokenAuthentication(
    token="XXXXX",  # Replace with your actual token
    server="XXXXX",  # Replace with your actual server URL
    skip_tls=False
)
auth.login()


## Scenario 1: RayJob to Existing Cluster

Let's create an existing cluster and submit a PyTorch training RayJob to it.


In [ ]:
# Create existing cluster for PyTorch training
print("=== Creating Existing Cluster for PyTorch Training ===")

# Define PyTorch training resource requirements
training_config = ClusterConfiguration(
    name='pytorch-training-cluster',
    num_workers=2,
    head_cpu_requests='1',
    head_cpu_limits='1',
    head_memory_requests=4,
    head_memory_limits=4,
    worker_cpu_requests='2',
    worker_cpu_limits='2',
    worker_memory_requests=8,
    worker_memory_limits=8,
    worker_extended_resource_requests={'nvidia.com/gpu': 1},
    worker_extended_resource_limits={'nvidia.com/gpu': 1}
)

# Use the configuration to create cluster
cluster = Cluster(training_config)
cluster.apply()

print("\n🚀 Cluster submitted! Waiting for it to be ready...")
cluster.wait_ready()

print("\n✅ Cluster is ready! Status:")
cluster.status()


In [ ]:
# Submit PyTorch training RayJob to existing cluster
print("=== Submitting PyTorch Training RayJob to Existing Cluster ===")

# Create RayJob with PyTorch training script
rayjob_existing = RayJob(
    job_name="pytorch-training-existing",
    cluster_name="pytorch-training-cluster",
    namespace="default",
    entrypoint="python pytorch_training_with_checkpointing.py",
    runtime_env={
        "pip": ["torch>=2.0.0", "torchvision", "numpy"],
        "env_vars": {
            "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:128",
            "CUDA_VISIBLE_DEVICES": "0"
        }
    },
    shutdown_after_job_finishes=False,  # Keep cluster running
    ttl_seconds_after_finished=300
)

print("\n📋 RayJob Configuration:")
print(f"  Job name: pytorch-training-existing")
print(f"  Cluster: pytorch-training-cluster")
print(f"  Entrypoint: python pytorch_training_with_checkpointing.py")
print(f"  Runtime environment: PyTorch 2.0+, torchvision, numpy")

# Submit the job
print("\n🚀 Submitting RayJob...")
submission_result = rayjob_existing.submit()
print(f"RayJob submitted successfully: {submission_result}")


## Manual Preemption Testing

To test how the checkpointing works with preemption, you can manually suspend the RayJob.

The following steps demonstrate how to test suspension and checkpointing:
1. Wait for the job to start running (check status)
2. Manually suspend the RayJob using kubectl or oc commands
3. Check job status - it should show SUSPENDED
4. Resume the job using kubectl or oc commands
5. The training will resume from the latest checkpoint

In [ ]:
# Check job status before suspension
status, ready = preemptible_job.status()
print(f"Job status before suspension: {status}")
print(f"Job ready: {ready}")

print("\n2. Wait for job to start running, then suspend:")
print("   # Wait until status shows RUNNING")
print("   # Then run one of these commands:")


### Suspend the RayJob

In [ ]:
!oc patch rayjob pytorch-training-preemptible -p '{\"spec\":{\"suspend\":true}}'

In [ ]:
# Check job status after suspension

status, ready = preemptible_job.status()
print(f"Job status after suspension: {status}")
print(f"Job ready: {ready}")
print(f"Expected: SUSPENDED")


Check that checkpoint was saved:
The training script saves checkpoints every epoch
You can verify this by checking the pod logs

### Resume the RayJob

In [ ]:
!oc patch rayjob pytorch-training-preemptible -p '{\"spec\":{\"suspend\":false}}'

In [ ]:
# Check job status after resume
status, ready = preemptible_job.status()
print(f"Job status after resume: {status}")
print(f"Job ready: {ready}")
print(f"Expected: RUNNING (training will resume from checkpoint)")


## Ray Dashboard forLog Monitoring

In [ ]:
# Get Ray Dashboard URL using CodeFlare SDK
print("Getting Ray Dashboard URL using CodeFlare SDK...")

try:
    # Use the native CodeFlare SDK function to get dashboard URI
    dashboard_url = cluster.cluster_dashboard_uri()
    
    if dashboard_url:
        print(f"\n🌐 Ray Dashboard Access:")
        print(f"Dashboard URL: {dashboard_url}")
        print(f"\n📋 To access the Ray Dashboard:")
        print(f"1. Open your browser and go to:")
        print(f"   {dashboard_url}")
    else:
        print("No dashboard URL found - cluster may not be ready yet")
        
except Exception as e:
    print(f"Failed to get dashboard URL: {e}")
    print("Make sure the cluster is ready and running")


## Cleanup

Finally, let's clean up our resources.


In [ ]:
# Clean up resources
print("=== Cleaning Up Resources ===")
# Take down the existing cluster
print("Taking down existing cluster...")
cluster.down()
print("\n✅ Cleanup completed!")


## Conclusion

This notebook demonstrated comprehensive PyTorch training with the CodeFlare SDK:

### **Key Features Demonstrated:**

1. **PyTorch Training with Checkpointing**:
   - Automatic checkpoint saving every epoch
   - Checkpoint loading and resume capability
   - Best practice checkpoint management

2. **Preemption Handling**:
   - RayJob suspension detection
   - Graceful checkpoint saving before preemption
   - Training resume from checkpoints

3. **Multiple Deployment Scenarios**:
   - Existing cluster: For persistent training infrastructure
   - Lifecycled cluster: For ephemeral training jobs
   - Both scenarios support preemption handling

### **Best Practices Shown:**

- **Checkpointing**: Save checkpoints regularly and before preemption
- **Error Handling**: Graceful handling of preemption and suspension
- **Monitoring**: Comprehensive job status tracking
- **Cleanup**: Proper resource cleanup after training

### **RayJob Preemption Support:**

The CodeFlare SDK supports RayJob preemption through:
- **SUSPENDED status**: Detects when jobs are preempted
- **Checkpoint integration**: Seamless resume from checkpoints
- **Both deployment modes**: Works with existing and lifecycled clusters

This provides a robust foundation for production PyTorch training workloads that can handle preemption and resource constraints gracefully.
stopping and sat